# NB 14 — Régression logistique Lasso multinomiale pure

## Objet

Tester le baseline **le plus simple possible** : régression logistique multinomiale avec pénalisation Elastic Net (`glmnet`, `family = "multinomial"`), directement sur la concaténation `[X_GE, X_CGH]`, sans aucune réduction de dimension préalable.

## Hypothèse à vérifier

Cooperative learning à ρ=0 dégénère théoriquement en **Lasso multi-bloc** sur la concaténation des blocs. Si NB14 reproduit le score de NB11 (~0.83), on confirme que :

1. La performance de NB11 vient **entièrement** de la sélection sparse supervisée Lasso
2. La machinerie cooperative (agrément cross-blocs) **n'apporte rien** sur ce dataset
3. Le pipeline le plus simple (Lasso multinomial direct) atteint le même plafond performance

## Différence avec NB11 et NB13

| Notebook | Sélection | Classifieur | Backend |
|---|---|---|---|
| NB11 (Coop argmax) | Cooperative L1 OvR (3 modèles binomiaux) | argmax | multiview |
| NB13 (sPCA+LogReg) | Sparse PCA non-supervisée + L1 | multinomial | mixOmics + glmnet |
| **NB14 (Lasso direct)** | **L1 supervisé direct, multinomial natif** | **multinomial** | **glmnet** |

NB14 = NB13 sans Sparse PCA. Si NB14 ≈ NB11, on confirme l'hypothèse.

## Hyperparamètres testés

- α ∈ {0, 0.5, 1} : Ridge / Elastic Net / Lasso
- λ : choisi par CV interne `cv.glmnet` (10-fold par défaut)

Comme dans NB13, mais sur **features brutes** au lieu de composantes Sparse PCA.

## 1. Setup

In [1]:
set.seed(42)
SEED <- 42
LABEL_ORDER <- c("cort", "dipg", "midl")

required_packages <- c("glmnet", "data.table", "caret")
to_install <- required_packages[!vapply(required_packages, requireNamespace,
                                        logical(1), quietly = TRUE)]
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}

suppressPackageStartupMessages({
  library(glmnet)
  library(data.table)
  library(caret)
})

cat("glmnet :", as.character(packageVersion("glmnet")), "\n")


glmnet : 5.0 


## 2. Chargement des données (identique aux autres notebooks)

In [2]:
root <- normalizePath(file.path(getwd()), winslash = "/", mustWork = FALSE)
data_dir <- if (dir.exists(file.path(root, "data"))) file.path(root, "data") else file.path(dirname(root), "data")

to_numeric_frame <- function(df) {
  rn <- rownames(df)
  out <- as.data.frame(
    lapply(df, function(x) as.numeric(gsub(",", ".", as.character(x), fixed = TRUE))),
    check.names = FALSE
  )
  rownames(out) <- rn
  out
}

extract_id_column <- function(df) if ("row_id" %in% names(df)) "row_id" else names(df)[1]

load_block <- function(block_name, split) {
  path <- file.path(data_dir,
    sprintf("ge_cgh_locIGR__multiblocks__%s__%s.csv", block_name, split))
  df <- as.data.frame(data.table::fread(path, check.names = FALSE))
  id_col <- extract_id_column(df)
  rownames(df) <- as.character(df[[id_col]])
  df[[id_col]] <- NULL
  to_numeric_frame(df)
}

load_targets <- function(split) {
  path <- file.path(data_dir,
    sprintf("ge_cgh_locIGR__multiblocks__y__%s.csv", split))
  y_df <- as.data.frame(data.table::fread(path, check.names = FALSE))
  id_col <- extract_id_column(y_df)
  rownames(y_df) <- as.character(y_df[[id_col]])
  y_df[[id_col]] <- NULL
  targets <- factor(
    LABEL_ORDER[max.col(as.matrix(y_df[, LABEL_ORDER]), ties.method = "first")],
    levels = LABEL_ORDER
  )
  names(targets) <- rownames(y_df)
  targets
}

fill_missing_from_train <- function(train_df, test_df) {
  medians <- vapply(train_df, median, numeric(1), na.rm = TRUE)
  for (col in names(train_df)) {
    train_df[[col]][is.na(train_df[[col]])] <- medians[[col]]
    test_df[[col]][is.na(test_df[[col]])] <- medians[[col]]
  }
  list(train = train_df, test = test_df)
}

X_ge_train  <- load_block("GE",  "train")
X_ge_test   <- load_block("GE",  "test")
X_cgh_train <- load_block("CGH", "train")
X_cgh_test  <- load_block("CGH", "test")
y_train     <- load_targets("train")
y_test      <- load_targets("test")

train_ids <- Reduce(intersect, list(rownames(X_ge_train), rownames(X_cgh_train), names(y_train)))
test_ids  <- Reduce(intersect, list(rownames(X_ge_test),  rownames(X_cgh_test),  names(y_test)))

X_ge_train  <- as.matrix(X_ge_train [train_ids, , drop = FALSE])
X_cgh_train <- as.matrix(X_cgh_train[train_ids, , drop = FALSE])
y_train     <- y_train [train_ids]
X_ge_test   <- as.matrix(X_ge_test  [test_ids,  , drop = FALSE])
X_cgh_test  <- as.matrix(X_cgh_test [test_ids,  , drop = FALSE])
y_test      <- y_test   [test_ids]

filled_ge  <- fill_missing_from_train(as.data.frame(X_ge_train), as.data.frame(X_ge_test))
X_ge_train <- as.matrix(filled_ge$train);  X_ge_test <- as.matrix(filled_ge$test)
filled_cgh <- fill_missing_from_train(as.data.frame(X_cgh_train), as.data.frame(X_cgh_test))
X_cgh_train <- as.matrix(filled_cgh$train); X_cgh_test <- as.matrix(filled_cgh$test)

# Concaténation directe — pas de Sparse PCA
X_train <- cbind(X_ge_train, X_cgh_train)
X_test  <- cbind(X_ge_test,  X_cgh_test)
colnames(X_train)[1:ncol(X_ge_train)]                                 <- paste0("GE__",  colnames(X_ge_train))
colnames(X_train)[(ncol(X_ge_train)+1):ncol(X_train)]                 <- paste0("CGH__", colnames(X_cgh_train))
colnames(X_test)  <- colnames(X_train)

cat(sprintf("Train: %d patients | Test: %d patients\n", length(y_train), length(y_test)))
cat(sprintf("X_train concaténé : %d × %d (GE: %d + CGH: %d)\n",
            nrow(X_train), ncol(X_train), ncol(X_ge_train), ncol(X_cgh_train)))


Train: 39 patients | Test: 14 patients
X_train concaténé : 39 × 16931 (GE: 15702 + CGH: 1229)


## 3. Sanity check — un seul fit Lasso multinomial sur tout le train

On utilise `cv.glmnet` qui sélectionne automatiquement λ par CV interne 10-fold. `type.multinomial = "grouped"` force la sparsité au niveau **variable** (un coefficient simultanément non-nul ou nul pour les 3 classes, plus interprétable).

In [3]:
set.seed(SEED)
fit_test <- cv.glmnet(
  x = X_train,
  y = y_train,
  family = "multinomial",
  type.multinomial = "grouped",
  alpha  = 1.0,                # Lasso pur
  nfolds = 10,
  standardize = TRUE
)

cat(sprintf("Lasso pur (α=1) : lambda.min = %.5f | lambda.1se = %.5f\n",
            fit_test$lambda.min, fit_test$lambda.1se))

# Coefficients sélectionnés
coefs <- coef(fit_test, s = "lambda.min")
n_ge  <- 0; n_cgh <- 0
for (cl in names(coefs)) {
  cmat <- coefs[[cl]]
  feat_names <- rownames(cmat)[2:nrow(cmat)]   # retirer intercept
  beta <- cmat[2:nrow(cmat), 1]
  is_ge  <- grepl("^GE__",  feat_names)
  is_cgh <- grepl("^CGH__", feat_names)
  nz <- abs(beta) > 1e-8
  n_ge_cl  <- sum(nz & is_ge)
  n_cgh_cl <- sum(nz & is_cgh)
  cat(sprintf("  Classe %s : %d/15702 GE non-nuls, %d/1229 CGH non-nuls\n",
              cl, n_ge_cl, n_cgh_cl))
}

# Variables actives (union sur les 3 classes)
all_active <- character(0)
for (cl in names(coefs)) {
  cmat <- coefs[[cl]]
  feat_names <- rownames(cmat)[2:nrow(cmat)]
  beta <- cmat[2:nrow(cmat), 1]
  all_active <- union(all_active, feat_names[abs(beta) > 1e-8])
}
n_ge_union  <- sum(grepl("^GE__",  all_active))
n_cgh_union <- sum(grepl("^CGH__", all_active))
cat(sprintf("\nUnion sur les 3 classes : %d GE + %d CGH = %d variables\n",
            n_ge_union, n_cgh_union, length(all_active)))

# Prédiction test
pred_test <- predict(fit_test, newx = X_test, s = "lambda.min", type = "class")[, 1]
cm_test <- caret::confusionMatrix(
  factor(pred_test, levels = LABEL_ORDER),
  factor(y_test,    levels = LABEL_ORDER)
)
cat("\n=== Test (sanity check, α=1) ===\n")
print(cm_test$table)
cat(sprintf("\nAccuracy : %.3f | Balanced accuracy : %.3f\n",
            cm_test$overall["Accuracy"],
            mean(cm_test$byClass[, "Balanced Accuracy"])))


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground

Lasso pur (α=1) : lambda.min = 0.05697 | lambda.1se = 0.18228
  Classe cort : 27/15702 GE non-nuls, 0/1229 CGH non-nuls
  Classe dipg : 27/15702 GE non-nuls, 0/1229 CGH non-nuls
  Classe midl : 27/15702 GE non-nuls, 0/1229 CGH non-nuls

Union sur les 3 classes : 27 GE + 0 CGH = 27 variables

=== Test (sanity check, α=1) ===
          Reference
Prediction cort dipg midl
      cort    5    0    1
      dipg    0    6    2
      midl    0    0    0

Accuracy : 0.786 | Balanced accuracy : 0.773


## 4. Cross-validation 7-fold × 3 sur α (protocole identique à NB09-NB13)

Pour chaque α ∈ {0, 0.5, 1}, on fait 21 folds et on moyenne. Pas de Sparse PCA, juste `cv.glmnet` à chaque fold pour sélectionner λ.

⚠️ Beaucoup plus rapide qu'NB13 parce qu'on n'a pas l'overhead Sparse PCA. Compter **5-15 min**.

In [4]:
ALPHA_GRID <- c(0, 0.5, 1.0)
cv_results <- data.frame()

set.seed(SEED)
outer_folds <- caret::createMultiFolds(y_train, k = 7, times = 3)

for (alpha_val in ALPHA_GRID) {
  cat(sprintf("\n=== alpha = %.1f ===\n", alpha_val))
  fold_scores <- numeric(length(outer_folds))
  t0_alpha <- Sys.time()

  for (i in seq_along(outer_folds)) {
    tr_idx <- outer_folds[[i]]
    va_idx <- setdiff(seq_along(y_train), tr_idx)

    fit_fold <- tryCatch(
      cv.glmnet(x = X_train[tr_idx, , drop = FALSE],
                y = y_train[tr_idx],
                family = "multinomial", type.multinomial = "grouped",
                alpha = alpha_val, nfolds = 5,
                standardize = TRUE),
      error = function(e) { message("  fold ", i, " failed: ", conditionMessage(e)); NULL }
    )
    if (is.null(fit_fold)) { fold_scores[i] <- NA_real_; next }

    pred_va <- predict(fit_fold,
                       newx = X_train[va_idx, , drop = FALSE],
                       s = "lambda.min", type = "class")[, 1]
    cm <- caret::confusionMatrix(
      factor(pred_va, levels = LABEL_ORDER),
      factor(y_train[va_idx], levels = LABEL_ORDER)
    )
    fold_scores[i] <- mean(cm$byClass[, "Balanced Accuracy"], na.rm = TRUE)
  }

  elapsed <- difftime(Sys.time(), t0_alpha, units = "mins")
  cat(sprintf("  Bal_acc moy = %.3f ± %.3f  (%.1f min)\n",
              mean(fold_scores, na.rm = TRUE),
              sd(fold_scores, na.rm = TRUE),
              as.numeric(elapsed)))

  cv_results <- rbind(cv_results, data.frame(
    alpha = alpha_val,
    mean_bal_acc = mean(fold_scores, na.rm = TRUE),
    sd_bal_acc   = sd(fold_scores, na.rm = TRUE),
    n_folds      = sum(!is.na(fold_scores))
  ))
}

cat("\n========== RÉSULTATS CV ==========\n")
print(cv_results)

best_idx   <- which.max(cv_results$mean_bal_acc)
best_alpha <- cv_results$alpha[best_idx]
cat(sprintf("\n>>> Meilleur alpha : %.1f  (CV bal_acc = %.3f ± %.3f)\n",
            best_alpha,
            cv_results$mean_bal_acc[best_idx],
            cv_results$sd_bal_acc[best_idx]))



=== alpha = 0.0 ===


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground

  Bal_acc moy = 0.706 ± 0.088  (3.0 min)

=== alpha = 0.5 ===


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground

  Bal_acc moy = 0.781 ± 0.120  (0.2 min)

=== alpha = 1.0 ===


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground

  Bal_acc moy = 0.784 ± 0.096  (0.2 min)

========== RÉSULTATS CV ==========
  alpha mean_bal_acc sd_bal_acc n_folds
1   0.0    0.7055556 0.08773997      21
2   0.5    0.7813492 0.12015982      21
3   1.0    0.7835979 0.09585911      21

>>> Meilleur alpha : 1.0  (CV bal_acc = 0.784 ± 0.096)


## 5. Refit final au meilleur α + évaluation test

In [5]:
set.seed(SEED)

fit_final <- cv.glmnet(
  x = X_train, y = y_train,
  family = "multinomial", type.multinomial = "grouped",
  alpha = best_alpha, nfolds = 10,
  standardize = TRUE
)

cat(sprintf("Refit final : alpha=%.1f, lambda.min=%.5f\n",
            best_alpha, fit_final$lambda.min))

# Comptage variables actives par classe
coefs_final <- coef(fit_final, s = "lambda.min")
cat("\nVariables non-nulles par classe :\n")
for (cl in names(coefs_final)) {
  cmat <- coefs_final[[cl]]
  feat_names <- rownames(cmat)[2:nrow(cmat)]
  beta <- cmat[2:nrow(cmat), 1]
  is_ge  <- grepl("^GE__",  feat_names)
  is_cgh <- grepl("^CGH__", feat_names)
  nz <- abs(beta) > 1e-8
  cat(sprintf("  %s : %d GE + %d CGH = %d variables\n",
              cl, sum(nz & is_ge), sum(nz & is_cgh), sum(nz)))
}

# Prédiction test
pred_final <- predict(fit_final, newx = X_test, s = "lambda.min", type = "class")[, 1]
probs_final <- predict(fit_final, newx = X_test, s = "lambda.min", type = "response")[, , 1]

cat("\nProbabilités prédites :\n")
print(round(probs_final, 3))

cm_final <- caret::confusionMatrix(
  factor(pred_final, levels = LABEL_ORDER),
  factor(y_test,     levels = LABEL_ORDER)
)
cat("\n=== Matrice de confusion test ===\n")
print(cm_final$table)
cat(sprintf("\nAccuracy : %.3f\n", cm_final$overall["Accuracy"]))
cat(sprintf("Balanced accuracy : %.3f\n",
            mean(cm_final$byClass[, "Balanced Accuracy"])))
cat("\nBal_acc par classe :\n")
print(round(cm_final$byClass[, "Balanced Accuracy"], 3))


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground

Refit final : alpha=1.0, lambda.min=0.05697

Variables non-nulles par classe :
  cort : 27 GE + 0 CGH = 27 variables
  dipg : 27 GE + 0 CGH = 27 variables
  midl : 27 GE + 0 CGH = 27 variables

Probabilités prédites :
     cort  dipg  midl
P08 0.810 0.079 0.111
P09 0.886 0.057 0.057
P11 0.027 0.529 0.444
P14 0.169 0.506 0.325
P19 0.017 0.786 0.197
P21 0.532 0.306 0.162
P24 0.075 0.602 0.323
P28 0.916 0.047 0.037
P29 0.020 0.970 0.011
P39 0.775 0.105 0.120
P42 0.115 0.808 0.077
P44 0.007 0.934 0.058
P47 0.926 0.012 0.061
P51 0.016 0.962 0.022

=== Matrice de confusion test ===
          Reference
Prediction cort dipg midl
      cort    5    0    1
      dipg    0    6    2
      midl    0    0    0

Accuracy : 0.786
Balanced accuracy : 0.773

Bal_acc par classe :
Class: cort Class: dipg Class: midl 
      0.944       0.875       0.500 


In [1]:
source(file.path(getwd(), "14c_multinomial_ungrouped.R"))

ERROR: Error: exists("X_train") is not TRUE


## 6. Comparaison directe avec NB09, NB11, NB13

| Méthode | Pipeline | CV bal_acc | Test bal_acc |
|---|---|---|---|
| NB09 SGCCA + LDA | SGCCA → LDA | 0.829 ± 0.133 | 0.924 |
| NB11 Coop argmax | Cooperative OvR ρ=0 → argmax | 0.833 ± 0.129 | 0.924 |
| NB13 sPCA + LogReg | Sparse PCA per-block → LogReg multinomial | 0.651 ± 0.111 | 0.727 |
| **NB14 Lasso direct** | **glmnet multinomial sur concat** | **(ce notebook)** | **(ce notebook)** |

In [6]:
nb09_cv      <- "0.829 ± 0.133";  nb09_test <- 0.924
nb11_cv      <- "0.833 ± 0.129";  nb11_test <- 0.924
nb13_cv      <- "0.651 ± 0.111";  nb13_test <- 0.727

nb14_cv_mean <- cv_results$mean_bal_acc[best_idx]
nb14_cv_sd   <- cv_results$sd_bal_acc[best_idx]
nb14_cv      <- sprintf("%.3f ± %.3f", nb14_cv_mean, nb14_cv_sd)
nb14_test    <- mean(cm_final$byClass[, "Balanced Accuracy"])

comparison <- data.frame(
  Notebook = c("NB09 SGCCA+LDA", "NB11 Coop argmax",
               "NB13 sPCA+LogReg",
               sprintf("NB14 Lasso direct (α=%.1f)", best_alpha)),
  CV_bal_acc   = c(nb09_cv, nb11_cv, nb13_cv, nb14_cv),
  Test_bal_acc = c(sprintf("%.3f", nb09_test),
                   sprintf("%.3f", nb11_test),
                   sprintf("%.3f", nb13_test),
                   sprintf("%.3f", nb14_test))
)
print(comparison, row.names = FALSE)

cat("\n========== DIAGNOSTIC ==========\n")
delta_vs_nb11 <- nb14_cv_mean - 0.833
cat(sprintf("Δ NB14 - NB11 (Cooperative ρ=0) : %+.3f\n", delta_vs_nb11))
if (abs(delta_vs_nb11) < 0.03) {
  cat("→ NB14 ≈ NB11 : confirmation que cooperative à ρ=0 = Lasso multinomial.\n")
  cat("→ La machinerie cooperative (agrément ρ) n'apporte rien.\n")
  cat("→ Lasso multinomial direct est suffisant et plus simple.\n")
} else if (delta_vs_nb11 > 0) {
  cat("→ NB14 > NB11 : Lasso multinomial bat même cooperative.\n")
  cat("→ La cooperative OvR (3 modèles binomiaux) est sous-optimale vs multinomial natif.\n")
} else {
  cat("→ NB14 < NB11 : la formulation OvR ou cooperative apporte quelque chose.\n")
}

delta_vs_nb13 <- nb14_cv_mean - 0.651
cat(sprintf("\nΔ NB14 - NB13 (sPCA + LogReg) : %+.3f\n", delta_vs_nb13))
cat("→ Quantifie l'apport de NE PAS faire de Sparse PCA non-supervisée en amont.\n")


                  Notebook    CV_bal_acc Test_bal_acc
            NB09 SGCCA+LDA 0.829 ± 0.133        0.924
          NB11 Coop argmax 0.833 ± 0.129        0.924
          NB13 sPCA+LogReg 0.651 ± 0.111        0.727
 NB14 Lasso direct (α=1.0) 0.784 ± 0.096        0.773

========== DIAGNOSTIC ==========
Δ NB14 - NB11 (Cooperative ρ=0) : -0.049
→ NB14 < NB11 : la formulation OvR ou cooperative apporte quelque chose.

Δ NB14 - NB13 (sPCA + LogReg) : +0.133
→ Quantifie l'apport de NE PAS faire de Sparse PCA non-supervisée en amont.


## 7. Lecture des résultats

Trois scénarios possibles :

### Scénario A — NB14 ≈ NB11 ≈ NB09 (≈ 0.83)

**Le plus attendu théoriquement.** Confirme que :
- Cooperative à ρ=0 = Lasso multi-bloc (équivalence mathématique)
- SGCCA et Lasso multinomial atteignent le même plafond performance
- Sur ce dataset, **le pipeline le plus simple** (Lasso multinomial direct, ~10 lignes de code) **suffit**

**Implication pour le rapport** : la performance est portée par la sélection sparse supervisée, indépendamment de la formulation algorithmique précise. La complexité méthodologique (cooperative, SGCCA) n'apporte pas de gain mesurable.

### Scénario B — NB14 ≈ NB11 mais < NB09

Improbable mais possible : SGCCA aurait un avantage spécifique grâce à sa formulation canonique (composantes orthogonales corrélées avec y), au-delà de la simple sélection L1.

### Scénario C — NB14 < NB11

Improbable : indiquerait que la formulation OvR + cooperative apporte quelque chose au-delà du multinomial natif. Surprenant — à creuser le cas échéant.

### Pour ton document de synthèse

Si scénario A (le plus probable), tu peux écrire la conclusion finale ultime :

> *"Quatre pipelines mathématiquement distincts (SGCCA + LDA, Cooperative OvR, Sparse PCA + LogReg multinomial, Lasso multinomial direct sur concaténation) ont été évalués sur protocole identique. Trois d'entre eux (SGCCA, Cooperative à ρ=0, Lasso multinomial direct) convergent vers la même balanced accuracy CV (~0.83). Seul Sparse PCA + LogReg chute à 0.65, faute de supervision lors de la sélection sparse. Cette équivalence empirique entre les trois méthodes supervisées sparses suggère que la performance est entièrement déterminée par la nature supervisée de la sélection L1, et non par les détails algorithmiques (canonical correlation analysis, agrément cross-bloc, formulation multinomiale vs OvR). Le pipeline le plus simple — Lasso multinomial direct sur la concaténation des blocs — est aussi le plus parcimonieux à présenter."*

C'est **la conclusion méthodologique la plus rigoureuse** possible pour ton rapport.


## 8. NB14b — OvR binomial Lasso direct (vraiment équivalent à NB11)

Le baseline **vraiment** équivalent à NB11 (cooperative ρ=0) :
- 3 régressions logistiques **binomiales** indépendantes (`family = "binomial"`)
- Chacune avec son **propre** λ optimal sélectionné par `cv.glmnet`
- Argmax des 3 probabilités sigmoïdes

Si NB14b ≈ NB11 (~0.83), on confirme empiriquement que la machinerie cooperative est inutile.

In [ ]:
# OvR : 3 fits binomiaux indépendants
fit_ovr_binomial <- function(X, y, alpha = 1.0) {
  K <- nlevels(y)
  classes <- levels(y)
  y_oh <- model.matrix(~ y - 1)
  colnames(y_oh) <- classes
  
  fits <- list()
  for (k in seq_len(K)) {
    fits[[classes[k]]] <- cv.glmnet(
      x = X, y = y_oh[, k],
      family = "binomial",
      alpha = alpha, nfolds = 5,
      standardize = TRUE
    )
  }
  list(classes = classes, fits = fits)
}

predict_ovr_argmax <- function(ovr_obj, X_new) {
  K <- length(ovr_obj$fits)
  probs <- matrix(0, nrow(X_new), K)
  colnames(probs) <- ovr_obj$classes
  for (k in seq_len(K)) {
    probs[, k] <- as.numeric(
      predict(ovr_obj$fits[[k]], newx = X_new, s = "lambda.min", type = "response")
    )
  }
  pred <- factor(ovr_obj$classes[apply(probs, 1, which.max)],
                 levels = ovr_obj$classes)
  list(class = pred, probs = probs)
}

# CV 7x3 sur OvR Lasso (alpha=1)
set.seed(SEED)
outer_folds_b <- caret::createMultiFolds(y_train, k = 7, times = 3)
scores_ovr <- numeric(length(outer_folds_b))

t0 <- Sys.time()
for (i in seq_along(outer_folds_b)) {
  tr_idx <- outer_folds_b[[i]]
  va_idx <- setdiff(seq_along(y_train), tr_idx)
  
  ovr_fold <- tryCatch(
    fit_ovr_binomial(X_train[tr_idx, , drop = FALSE], y_train[tr_idx], alpha = 1.0),
    error = function(e) NULL
  )
  if (is.null(ovr_fold)) { scores_ovr[i] <- NA; next }
  
  pred_va <- predict_ovr_argmax(ovr_fold, X_train[va_idx, , drop = FALSE])$class
  cm <- caret::confusionMatrix(pred_va,
                                factor(y_train[va_idx], levels = LABEL_ORDER))
  scores_ovr[i] <- mean(cm$byClass[, "Balanced Accuracy"], na.rm = TRUE)
}
cat(sprintf("NB14b OvR Lasso : CV bal_acc = %.3f ± %.3f  (%.1f min)\n",
            mean(scores_ovr, na.rm = TRUE),
            sd(scores_ovr, na.rm = TRUE),
            as.numeric(difftime(Sys.time(), t0, units = "mins"))))

# Refit final OvR + test
ovr_final <- fit_ovr_binomial(X_train, y_train, alpha = 1.0)

# lambda.min par classe
cat("\nLambda.min par classe :\n")
for (cl in ovr_final$classes) {
  fit_k <- ovr_final$fits[[cl]]
  beta_k <- coef(fit_k, s = "lambda.min")
  feat_names <- rownames(beta_k)[2:nrow(beta_k)]
  is_ge  <- grepl("^GE__",  feat_names)
  is_cgh <- grepl("^CGH__", feat_names)
  nz <- abs(beta_k[2:nrow(beta_k), 1]) > 1e-8
  cat(sprintf("  %s : lambda.min=%.4f, %d GE + %d CGH actives\n",
              cl, fit_k$lambda.min, sum(nz & is_ge), sum(nz & is_cgh)))
}

pred_test_ovr <- predict_ovr_argmax(ovr_final, X_test)
cat("\nProbabilités OvR test set :\n")
print(round(pred_test_ovr$probs, 3))

cm_ovr <- caret::confusionMatrix(pred_test_ovr$class,
                                  factor(y_test, levels = LABEL_ORDER))
cat("\n=== Matrice de confusion test (NB14b OvR Lasso) ===\n")
print(cm_ovr$table)
cat(sprintf("\nAccuracy : %.3f\n", cm_ovr$overall["Accuracy"]))
cat(sprintf("Balanced accuracy : %.3f\n",
            mean(cm_ovr$byClass[, "Balanced Accuracy"])))

cat("\n========== COMPARAISON NB11 vs NB14a vs NB14b ==========\n")
cat(sprintf("NB11  Cooperative OvR (ρ=0)   : 0.833 ± 0.129 CV | midl 2/3\n"))
cat(sprintf("NB14a Multinomial Lasso       : %.3f ± %.3f CV | midl 0/3\n",
            nb14_cv_mean, nb14_cv_sd))
cat(sprintf("NB14b OvR binomial Lasso      : %.3f ± %.3f CV\n",
            mean(scores_ovr, na.rm = TRUE),
            sd(scores_ovr, na.rm = TRUE)))

if (abs(mean(scores_ovr, na.rm = TRUE) - 0.833) < 0.02) {
  cat("\n→ NB14b ≈ NB11 : CONFIRMÉ que cooperative ρ=0 = OvR binomial Lasso.\n")
  cat("→ La machinerie cooperative (agrément ρ) n'apporte rien.\n")
  cat("→ La différence avec NB14a (multinomial) vient du mécanisme OvR + argmax.\n")
}
